In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

MODEL_NAME   = "openai/gpt-oss-120b"
MODEL_FAMILY = "groq"

# psychscanner automatically loads GROQ_API_KEY from the environment
# set it in your .env file or shell before running
from langchain.chat_models import init_chat_model

llm = init_chat_model(model=MODEL_NAME, model_provider=MODEL_FAMILY, temperature=0)
llm.invoke("what's your name")

AIMessage(content='I’m ChatGPT—here to help with any questions you have!', additional_kwargs={'reasoning_content': 'The user asks "what\'s your name". We should respond with a friendly answer. According to policy, we can give name: ChatGPT. No disallowed content.'}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 75, 'total_tokens': 133, 'completion_time': 0.123358119, 'completion_tokens_details': {'reasoning_tokens': 35}, 'prompt_time': 0.003314478, 'prompt_tokens_details': None, 'queue_time': 0.004650858, 'total_time': 0.126672597}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_dc4ca88fa4', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e02f1-49c8-70a1-82c3-c90f346d4d35-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 58, 'total_tokens': 133, 'output_token_details': {'reasoning': 35}})

In [2]:
from langchain_core.messages import HumanMessage
from psychscanner import ExpCard, ExpCardInit, ScannerModel
from psychscanner.parsers import (
    list_parsers,
    get_parser,
    ResponseRmStEI,
    AllResponseRMEI,
)
from psychscanner.datasets.prompts.parser_tasks import (
    Response_part_1_rm,
    Response_part_2_rm,
)


/opt/anaconda3/envs/psyscan/lib/python3.11/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
import json
import shutil
from pathlib import Path
from langchain_core.messages import HumanMessage


TASKS_DIR = Path.cwd() / 'tasks'
RUN_DIR   = Path.cwd() / '_rm_tutorial_runs'
# if RUN_DIR.exists():
#     shutil.rmtree(RUN_DIR)

    
task_files = {
    'singleturn'  : TASKS_DIR / 'rm_singleturn_demo.json',
    'trialchain'  : TASKS_DIR / 'rm_trialchain_demo.json',
    'episodic'    : TASKS_DIR / 'rm_episodic_demo.json',
    'episodic_fb' : TASKS_DIR / 'rm_episodic_fb_demo.json',
}
for name, p in task_files.items():
    print(f'  {name:13} -> {p.name}  (exists={p.exists()})')

  singleturn    -> rm_singleturn_demo.json  (exists=True)
  trialchain    -> rm_trialchain_demo.json  (exists=True)
  episodic      -> rm_episodic_demo.json  (exists=True)
  episodic_fb   -> rm_episodic_fb_demo.json  (exists=True)


In [4]:
def make_card(variant, *, memory, chain_type, parser='1', feedback=False, feedback_fn=None):
    """Build an ExpCardInit for the given tutorial variant.

    Parameters
    ----------
    parser : str
        '1'       → resolve class name from task JSON via the parsers registry.
        'dynamic' → use built-in RM routing (Response_part_1_rm for encoding
                    trials, Response_part_2_rm for test trials).
    """
    card = ExpCardInit()
    card.proj_dir       = RUN_DIR / variant
    card.projectname    = variant + "23200000"
    card.model          = MODEL_NAME
    card.family         = MODEL_FAMILY
    card.parameters     = {'temperature': 0}
    card.task_file      = task_files[variant]
    card.parser         = parser
    card.cogtype        = 'no'
    card.nsim           = 1
    card.tunnel_status  = '0'
    card.memory         = memory
    card.chain_type     = chain_type
    card.feedback       = feedback
    card.feedback_fn    = feedback_fn
    return card


def parse_pred_resp(pred_resp):
    """Decode the stringified-dict AIMessage content back to a Python object."""
    content = pred_resp.content if hasattr(pred_resp, 'content') else str(pred_resp)
    try:
        return ast.literal_eval(content)
    except Exception:
        return {'_raw': content[:120]}


def show_trials(trials, label=''):
    if label:
        print(f'--- {label} ---')
    print(f'  {"trcode":15}  parsed response')
    print(f'  {"-"*15}  {"-"*65}')
    for t in trials:
        parsed = parse_pred_resp(t['pred_resp'])
        print(f'  {t["trcode"]:15}  {parsed}')

In [5]:
card_v1  = make_card('singleturn', memory='SingleTurn', chain_type='item', parser='1')
exp_v1   = ExpCard(card_v1)
print(f'Resolved parser: {exp_v1.parser.__name__}  (module: {exp_v1.parser.__module__.split(".")[-1]})')
scanner_v1 = ScannerModel(expcard=exp_v1)
trials_v1  = scanner_v1.run()[0]
print()
show_trials(trials_v1, label='V1 — singleturn / item / no feedback')

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_rm_tutorial_runs/singleturn


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_rm_tutorial_runs/singleturn/singleturn23200000/zs_rm_2op/groq_openai/gpt-oss-120b_SingleTurn


----<>----


Resolved parser: ResponseRmStEI  (module: parser_tasks)
--<api key>-- loaded GROQ_API_KEY from environment


...

4it [00:03,  1.18it/s]


2026-05-07 10:57:18.586 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-05-07 10:57:18.591 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END



--- V1 — singleturn / item / no feedback ---
  trcode           parsed response
  ---------------  -----------------------------------------------------------------
  imagined_1       {'_raw': "{'Word_2': 'abrasive', 'Rating': 90.0, 'Judgment': 'internal', 'Confidence': 6}"}
  perceived_2      {'_raw': "{'Word_2': 'sweet', 'Rating': 92.0, 'Judgment': 'external', 'Confidence': 6}"}
  imagined_3       {'_raw': "{'Word_2': 'garden', 'Rating': 55.0, 'Judgment': 'internal', 'Confidence': 6}"}
  perceived_4      {'_raw': "{'Word_2': 'cushion', 'Rating': 85.0, 'Judgment': 'external', 'Confidence': 6}"}
